# BGE-M3 Dense structured (업무 필터 없음)

검색평가대상 121개 질문을 BGE-M3로 임베딩하고, `KDIC_output.zip`의 427개 청크 벡터에서 Top-10을 검색한 뒤 Gold 청크와 비교합니다.

이 노트북은 답변 생성 LLM을 호출하지 않습니다. HCX 임베딩 API만 사용합니다.

## 준비 파일

1. `KDIC_BGE_M3_Dense_평가기.zip`
2. `KDIC_output.zip`
3. `Evaluation_DataSet_v3.5.xlsx`

API 키는 코드에 작성하지 말고 Colab 왼쪽의 **열쇠 아이콘 → 새 보안 비밀 추가**에서 이름을 `HCX_API_KEY`로 등록하세요.

In [ ]:
%pip -q install "numpy>=1.26" "pandas>=2.1" "openpyxl>=3.1"

import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import pandas as pd
from IPython.display import display

WORK_ROOT = Path('/content/kdic_dense_evaluation')
EVALUATOR_ROOT = WORK_ROOT / 'evaluator'
RESULT_ROOT = WORK_ROOT / 'results'
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print('환경 준비 완료:', WORK_ROOT)

## 1. 파일 업로드

아래 셀을 실행하고 준비한 파일 3개를 한 번에 선택합니다.

In [ ]:
from google.colab import files

uploaded = files.upload()
for filename, content in uploaded.items():
    target = WORK_ROOT / filename
    target.write_bytes(content)
    print(f'업로드: {target.name} ({target.stat().st_size:,} bytes)')

In [ ]:
dataset_candidates = list(WORK_ROOT.glob('*.xlsx'))
kdic_candidates = [
    p for p in WORK_ROOT.glob('*.zip')
    if p.name == 'KDIC_output.zip' or p.name.startswith('KDIC_output (')
]
evaluator_candidates = [
    p for p in WORK_ROOT.glob('*.zip')
    if 'Dense' in p.name or '평가기' in p.name
]

if not dataset_candidates:
    raise RuntimeError('평가 XLSX를 찾을 수 없습니다.')
if not kdic_candidates:
    raise RuntimeError('KDIC_output.zip을 찾을 수 없습니다.')
if not evaluator_candidates:
    raise RuntimeError('평가기 ZIP을 찾을 수 없습니다.')

def choose_uploaded(candidates, preferred_name):
    preferred = WORK_ROOT / preferred_name
    if preferred in candidates:
        return preferred
    return max(candidates, key=lambda path: path.stat().st_mtime)

DATASET_PATH = choose_uploaded(
    dataset_candidates,
    'Evaluation_DataSet_v3.5.xlsx',
)
KDIC_ZIP_PATH = choose_uploaded(kdic_candidates, 'KDIC_output.zip')
EVALUATOR_ZIP_PATH = choose_uploaded(
    evaluator_candidates,
    'KDIC_BGE_M3_Dense_Structured_평가기.zip',
)

if EVALUATOR_ROOT.exists():
    shutil.rmtree(EVALUATOR_ROOT)
EVALUATOR_ROOT.mkdir(parents=True)
with zipfile.ZipFile(EVALUATOR_ZIP_PATH) as archive:
    archive.extractall(EVALUATOR_ROOT)

EVALUATOR_SCRIPT = EVALUATOR_ROOT / 'evaluate_bge_m3_dense_structured.py'
if not EVALUATOR_SCRIPT.exists():
    raise FileNotFoundError(EVALUATOR_SCRIPT)

print('평가데이터셋:', DATASET_PATH.name)
print('검색 데이터:', KDIC_ZIP_PATH.name)
print('평가기:', EVALUATOR_SCRIPT)

## 2. API 키 불러오기

Colab Secrets에 `HCX_API_KEY`가 있으면 자동으로 사용합니다. 등록하지 않았다면 화면에 표시되지 않는 입력창이 나타납니다.

In [ ]:
import getpass
from google.colab import userdata

try:
    api_key = userdata.get('HCX_API_KEY')
except Exception:
    api_key = None

if not api_key:
    api_key = getpass.getpass('HCX_API_KEY를 입력하세요: ')

api_key = str(api_key).strip()
if not api_key:
    raise RuntimeError('HCX_API_KEY가 비어 있습니다.')
if api_key.lower().startswith('bearer '):
    raise RuntimeError("키 앞에 'Bearer '를 붙이지 마세요.")

os.environ['HCX_API_KEY'] = api_key
del api_key
print('API 키 등록 완료(값은 출력하지 않음)')

## 3. Dry-run

API를 호출하지 않고 질문·청크·임베딩·Gold 연결을 먼저 검사합니다.

In [ ]:
dry_result_dir = WORK_ROOT / 'results_dry'
dry_command = [
    sys.executable,
    str(EVALUATOR_SCRIPT),
    '--dataset', str(DATASET_PATH),
    '--sheet-name', '평가데이터셋 v3',
    '--kdic-zip', str(KDIC_ZIP_PATH),
    '--output-dir', str(dry_result_dir),
    '--no-domain-filter',
    '--dry-run',
]
subprocess.run(dry_command, cwd=EVALUATOR_ROOT, check=True)

## 4. BGE-M3 Dense 전체 평가

검색평가대상 121개 질문의 임베딩을 순서대로 생성합니다. 질문 임베딩은 결과 폴더에 캐시되므로 중간에 중단되어도 재실행 시 이미 처리한 질문은 API를 다시 호출하지 않습니다.

In [ ]:
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
eval_command = [
    sys.executable,
    str(EVALUATOR_SCRIPT),
    '--dataset', str(DATASET_PATH),
    '--sheet-name', '평가데이터셋 v3',
    '--kdic-zip', str(KDIC_ZIP_PATH),
    '--output-dir', str(RESULT_ROOT),
    '--no-domain-filter',
]
subprocess.run(eval_command, cwd=EVALUATOR_ROOT, check=True)

## 5. 평가 결과 확인

In [ ]:
summary = json.loads((RESULT_ROOT / 'summary.json').read_text(encoding='utf-8'))
overall = pd.DataFrame([summary['overall']])
by_domain = pd.read_csv(RESULT_ROOT / 'summary_by_domain.csv')
questions = pd.read_csv(RESULT_ROOT / 'question_results.csv')

print('전체 평가 결과')
display(overall)
print('\n도메인별 평가 결과')
display(by_domain)
print('\nHit@3 실패 질문 예시')
display(
    questions.loc[
        questions['hit_at_3'] == 0,
        ['evaluation_id', 'question', 'domain', 'gold_chunk_ids', 'retrieved_chunk_ids'],
    ].head(10)
)

## 6. 결과 다운로드

In [ ]:
archive_path = shutil.make_archive(
    '/content/KDIC_BGE_M3_Dense_Structured_업무필터없음_평가결과',
    'zip',
    RESULT_ROOT,
)
print('결과 압축:', archive_path)
files.download(archive_path)